# 06 - Localisation and statistical baselines

**Owner:** Person 4


Three non-MIL views of the problem: a patch-level CNN probability map, the
attention-derived statistical features with ordinal logistic regression /
XGBoost baselines, and the localisation overlap scores that put a number on
"the heatmap looks right".


## 0. Colab setup (run first)

Mounts Drive, clones the repo, symlinks `data/features` and `results` to Drive so a
VM recycle does not lose a run, installs `h5py`, and prints a provenance hash you can
compare with your teammates. Re-run it after any restart.


In [ ]:
# ============================================================================
# COLAB SETUP -- run this cell first, in every session.
# It is idempotent: safe to re-run after a VM restart.
# ============================================================================
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO = Path("/content/prostate-mil-project")
    DATA = Path("/content/drive/MyDrive/prostate_mil_data")   # features live here (~600 MB/encoder)

    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/koushikkkkkkkkkk/prostate-mil-project.git", str(REPO)], check=True)

    DATA.mkdir(parents=True, exist_ok=True)
    (DATA / "results").mkdir(parents=True, exist_ok=True)

    # features + results survive VM recycling only if they live on Drive
    for link, target in (("data/features", DATA / "features"), ("results", DATA / "results")):
        link_path = REPO / link
        if not link_path.exists() and not link_path.is_symlink():
            link_path.parent.mkdir(parents=True, exist_ok=True)
            os.symlink(target, link_path)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "h5py"], check=False)
    os.chdir(REPO)
    sys.path.insert(0, str(REPO))
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(REPO)
    sys.path.insert(0, str(REPO))

import torch
n_bags = len(list((REPO / "data" / "features").glob("*.pt")))
print("repo   :", REPO)
print("torch  :", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("features on disk:", n_bags, "bags")
if n_bags == 0:
    print("\n!! No feature bags found. Run this once (writes to Drive):")
    print("   python scripts/download_features_and_coords.py --manifest data/common_split.csv --out-dir data/features")
    print("   or point the symlink at an existing feature folder.")

# ---- provenance guard: everyone must train on byte-identical inputs ---------
import hashlib
manifest = REPO / "data" / "common_split.csv"
if manifest.exists():
    digest = hashlib.sha256(manifest.read_bytes()).hexdigest()[:16]
    print("split hash:", digest, "| slides:", sum(1 for _ in manifest.open()) - 1)


In [ ]:

# ---- hard guard: never report synthetic numbers ----------------------------
# Runs before any training cell in this notebook. Mock results are a UI smoke
# test only and must not appear in the report.
try:
    MANIFEST
except NameError:
    from pathlib import Path as _P
    MANIFEST = _P("data/common_split.csv")
assert "mock" not in str(MANIFEST).lower(), (
    f"REFUSING TO TRAIN: MANIFEST={MANIFEST} is synthetic. "
    "Mock numbers are never reportable -- point it at data/common_split.csv."
)
print("provenance guard OK --", MANIFEST)


## Setup

In [ ]:
import os, sys, json
from pathlib import Path
os.environ.setdefault("MPLBACKEND", "Agg")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, torch
print("torch", torch.__version__)

## 1. Attention-vs-mask overlap (localisation score)

In [ ]:
from src.heatmaps import evaluate_localisation

# Uses any exported attention array + a mask image.
npz_dir = ROOT / "results/mock_heatmaps/arrays"
masks = ROOT / "data/mock/masks"
if npz_dir.exists() and masks.exists():
    import numpy as np
    example = sorted(npz_dir.glob("*.npz"))[0]
    payload = np.load(example, allow_pickle=True)
    mask_path = masks / f"{example.stem}.png"
    score = evaluate_localisation(payload["attention"], mask_path, payload["coords"], patch_size=256)
    print(example.stem, json.dumps(score, indent=2, default=float))
else:
    print("Run scripts/run_mock_demo.sh first to produce attention arrays.")

## 2. Statistical baselines over attention features

In [ ]:
from src.statistical_models import FEATURE_NAMES, OrdinalLogisticRegression, bag_statistics
print("x_stat features:", FEATURE_NAMES)

att_csv = next(iter(sorted((ROOT / "results").glob("mock_eval/attention_*.csv"))), None)
if att_csv:
    import subprocess, sys
    cmd = [sys.executable, "-m", "src.statistical_models",
           "--attention-csv", str(att_csv),
           "--manifest", str(ROOT / "data/mock/manifest.csv"),
           "--out-dir", str(ROOT / "results/mock_statistical")]
    res = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(res.stdout[-800:]); print(res.stderr[-500:])
else:
    print("No attention CSV yet.")

## 3. Why there is no patch-level classifier

PANDA ships no patch-level labels for this provider (the masks are 99.88 % background,
no Gleason pixels), and labelling tiles with the slide grade injects 70–90 % benign-stroma
label noise. Localisation is therefore reported from MIL attention only — see
`report/REPORT.md` §3.3. Nothing is trained in this section.


## Reporting notes

* Patch-level AUC/F1 requires patch labels that PANDA does not provide, so no patch classifier
  is therefore a weakly supervised *proxy* and its numbers are reported as such.
* Overlap scores are coarse: PANDA has no pixel-level ISUP masks, so they use
  tissue/tumour masks and are a sanity check, not a segmentation metric.
* The statistical baseline answers "is MIL needed at all?" -- report it next to
  ABMIL / CLAM-SB / TransMIL.